# Dataset RAG Chatbot

Reads metadata of datasets, embeds it, stores the vectors in ChromaDB, and exposes a small chatbot that answers questions using retrieved metadata.

Layout:

- `models.yaml` + `get_model.py` — swappable OpenAI-compatible backends (chat and embed)
- `base_agent.py` — `BaseAgent`; `DatasetBot` below subclasses it
- `vector_stores.py` — `ChromaStore` behind a `make_store(...)` factory (easy to extend later)

**Embeddings** default to NRP's `qwen3-embedding` (same `NRP_API_KEY` as the chat models). Set `EMBED_MODEL_KEY = None` to fall back to local `sentence-transformers` (no API key).

**Datasets** come from NDP Catalog.

## 0. Install dependencies

**Before running anything below**, install the project requirements. Run this once per environment:

```bash
pip install -r requirements.txt
```

Or run the cell below from inside the notebook:

In [ ]:
# Run once per environment; safe to skip on subsequent runs.
%pip install -q -r requirements.txt

## 0.5 Create your `.env`

The notebook reads `NRP_API_KEY` from a `.env` file in this directory (already covered by `.gitignore`).

Run the first cell once to create an empty `.env`. Then **edit the second cell**: replace `paste-your-key-here` with your actual NRP key and run it — it writes the key into `.env` via `cat` heredoc.

In [ ]:
!touch .env

In [ ]:
%%bash
cat > .env <<'EOF'
NRP_API_KEY=paste-your-key-here
EOF
echo 'Wrote .env:'
sed 's/=.*/=***/' .env

## 1. Setup

In [ ]:
import os
import json
from pathlib import Path

from dotenv import load_dotenv
load_dotenv(override=True)
print(repr(os.environ.get('NRP_API_KEY'))[:30])


from get_model import load_models_config, call_model, call_embedding, filter_by_task
from base_agent import BaseAgent
from vector_stores import make_store

config = load_models_config('models.yaml')
print('Available models:')
for key, entry in config.items():
    print(f'  [{entry["task"]:5s}] {key}  ->  {entry["model"]}')

## 2. Pick chat and embedding models

`CHAT_MODEL_KEY` — any entry in `models.yaml` with `task: chat`.

`EMBED_MODEL_KEY` — `None` = local sentence-transformers (no API key). Or an `embed/`-tasked key to use a `/v1/embeddings` endpoint.

In [ ]:
CHAT_MODEL_KEY = 'gemma'              # any entry in models.yaml with task: chat
EMBED_MODEL_KEY = 'qwen3-embedding'   # task: embed; set to None for local sentence-transformers

chat_cfg = config[CHAT_MODEL_KEY]
embed_cfg = config[EMBED_MODEL_KEY] if EMBED_MODEL_KEY else None

print(f'Chat:  {CHAT_MODEL_KEY} -> {chat_cfg["model"]} @ {chat_cfg["base_url"]}')
if embed_cfg is None:
    print('Embed: local sentence-transformers (all-MiniLM-L6-v2)')
else:
    print(f'Embed: {EMBED_MODEL_KEY} -> {embed_cfg["model"]} @ {embed_cfg["base_url"]}')

## 3. Load dataset metadata

Two sources are supported — set `DATASET_SOURCE` to pick one:

- **`'ndp'`** — pull fresh metadata from the NDP CKAN catalog by slug / UUID (`DATASET_IDS`).
- **`'local'`** — load metadata from files already on disk at `DATASETS_PATH`. That path can be:
  - a directory of `.json` files, one CKAN-style dataset per file, or
  - a directory of subfolders, each containing a `metadata.json` / `metadata.yaml`. If a subfolder has no metadata file, one is auto-synthesized from the folder name and its file listing.

In [ ]:
import requests
import yaml
from pathlib import Path
import pandas as pd

# --- Source 1: NDP CKAN catalog ---------------------------------------------
CATALOG_URL = 'https://nationaldataplatform.org/catalog'

def _ckan_get(action: str, params: dict = None) -> dict:
    url = f'{CATALOG_URL}/api/3/action/{action}'
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    if not data.get('success'):
        raise ValueError(f'CKAN error: {data.get("error")}')
    return data['result']

def fetch_dataset(dataset_id: str) -> dict:
    return _ckan_get('package_show', {'id': dataset_id})

def list_datasets(limit: int = 3) -> list:
    return _ckan_get('package_list', {'limit': limit})


# --- Source 2: local files ---------------------------------------------------
_IGNORED_DIRS = {'.ipynb_checkpoints', '__pycache__', '.git', '.DS_Store'}

def _real_subdirs(path: Path) -> list[Path]:
    return [
        p for p in path.iterdir()
        if p.is_dir() and not p.name.startswith('.') and p.name not in _IGNORED_DIRS
    ]


def load_local_datasets(path) -> list[dict]:
    """Load dataset metadata from a local directory. Accepts:

    1. A directory of `.json` files — each file is one CKAN-style dataset.
    2. A directory of `.csv` files (no real subdirs) — treated as a single
       dataset: data CSVs become resources, metric-description CSVs (header
       has `Metrics` + a `Description...` column) become `extras`.
    3. A directory of subfolders — each subfolder is one dataset (same rules
       as #2 inside each).

    Hidden / noise dirs (`.ipynb_checkpoints`, `__pycache__`, ...) are ignored.
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'DATASETS_PATH does not exist: {path}')
    if not path.is_dir():
        raise ValueError(f'DATASETS_PATH must be a directory, got: {path}')

    subdirs = _real_subdirs(path)
    json_files = sorted(path.glob('*.json'))
    csv_files = sorted(path.glob('*.csv'))

    # Flat JSON layout
    if json_files and not subdirs:
        return [json.loads(f.read_text()) for f in json_files]

    # Flat CSV layout: one folder = one dataset (data + metric CSVs side by side)
    if csv_files and not subdirs and not json_files:
        meta = _read_folder_metadata(path)
        return [meta] if meta else []

    # Subfolder-per-dataset layout
    results = []
    for folder in sorted(subdirs):
        meta = _read_folder_metadata(folder)
        if meta:
            results.append(meta)
    return results

def _filter_dataset_entries(results: list[dict]) -> list[dict]:
    # Read the Metrics column from dataset_keys_to_keep.csv — these are the
    # row-column names (or resource fields) we want to keep.
    keys_to_keep = set(pd.read_csv('dataset_keys_to_keep.csv')['Metrics'].dropna())

    def summary(resource):
        data = resource.get('data')
        if isinstance(data, list) and data and isinstance(data[0], dict):
            return f"{len(data)} rows x {len(data[0])} cols"
        return f"{len(resource)} fields"

    # Top-level CKAN fields the document builder consumes — preserve verbatim.
    passthrough_fields = ('name', 'title', 'notes', 'tags', 'organization', 'extras')

    filtered_results = []
    for entry in results:
        title = entry.get('title', 'N/A')
        before = [summary(r) for r in entry.get('resources', [])]
        print(f"Original '{title}': {len(entry.get('resources', []))} resource(s) — {before}")

        filtered_entry = {k: entry[k] for k in passthrough_fields if k in entry}
        filtered_entry['resources'] = []
        for resource in entry.get('resources', []):
            fr = {k: resource[k] for k in ('name', 'format') if k in resource}
            if 'data' in resource and isinstance(resource['data'], list):
                # CSV resource: filter columns of each row
                fr['data'] = [
                    {k: v for k, v in row.items() if k in keys_to_keep}
                    for row in resource['data']
                ]
            else:
                # Non-CSV resource: filter top-level resource fields
                fr.update({k: resource[k] for k in keys_to_keep if k in resource})
            filtered_entry['resources'].append(fr)
        filtered_results.append(filtered_entry)

        after = [summary(r) for r in filtered_entry['resources']]
        print(f"Filtered '{title}': {len(filtered_entry['resources'])} resource(s) — {after}")
    return filtered_results


def _is_metric_description_csv(path: Path) -> bool:
    """A CSV is a metric-description sheet if its header has a `Metrics` column
    plus at least one `Description...` column (case-insensitive)."""
    try:
        header = pd.read_csv(path, nrows=0).columns.tolist()
    except Exception:
        return False
    cols = [str(c).strip().lower() for c in header]
    return 'metrics' in cols and any('description' in c for c in cols)


def _metric_csv_to_extras(path: Path) -> list[dict]:
    """Turn a metric-description CSV into CKAN-style `extras` entries: one per
    metric, value = simplified description (falls back to detailed)."""
    if not _is_metric_description_csv(path):
        return []
    df = pd.read_csv(path)
    cols_lower = {c.lower(): c for c in df.columns}
    desc_col = next(
        (cols_lower[c] for c in cols_lower if 'description' in c and 'simpl' in c),
        next((cols_lower[c] for c in cols_lower if 'description' in c), None),
    )
    extras = []
    for _, row in df.iterrows():
        metric = row.get('Metrics')
        if pd.isna(metric):
            continue
        desc = row.get(desc_col, '') if desc_col else ''
        if pd.isna(desc) or not str(desc).strip():
            continue
        extras.append({'key': str(metric), 'value': str(desc).strip()})
    return extras


def _read_folder_metadata(folder: Path) -> dict | None:
    # Preferred metadata files
    meta = None
    if (folder / 'metadata.json').exists():
        meta = json.loads((folder / 'metadata.json').read_text())
    else:
        for name in ('metadata.yaml', 'metadata.yml'):
            if (folder / name).exists():
                meta = yaml.safe_load((folder / name).read_text())
                break
        if meta is None:
            for f in sorted(folder.glob('*.json')):
                meta = json.loads(f.read_text())
                break

    csvs = sorted(folder.glob('*.csv'))
    metric_csvs = [c for c in csvs if _is_metric_description_csv(c)]
    data_csvs = [c for c in csvs if c not in metric_csvs]

    # Synthesize a minimal entry if no JSON/YAML metadata was found.
    if meta is None:
        if not csvs:
            files = sorted(p.name for p in folder.iterdir() if p.is_file())
            if not files:
                return None
            return {
                'name': folder.name,
                'title': folder.name.replace('_', ' ').replace('-', ' ').title(),
                'notes': f'Local dataset at {folder}. Files: {", ".join(files)}',
                'resources': [
                    {'name': n, 'format': Path(n).suffix.lstrip('.').upper()} for n in files
                ],
            }
        meta = {
            'name': folder.name,
            'title': folder.name.replace('_', ' ').replace('-', ' ').title(),
            'resources': [],
        }

    # Attach data CSVs as resources (rows preserved as records).
    for csv_path in data_csvs:
        df = pd.read_csv(csv_path)
        meta.setdefault('resources', []).append({
            'name': csv_path.name,
            'format': 'CSV',
            'data': df.to_dict(orient='records'),
        })

    # Merge metric-description CSVs into extras so the bot sees per-column docs.
    for csv_path in metric_csvs:
        extras = _metric_csv_to_extras(csv_path)
        if extras:
            meta.setdefault('extras', []).extend(extras)

    return meta

## Let's use a local dataset for our first try 
Make sure you change the DATASET_SOURCE to **`'local'`** and set the DATASET_PATH to **`'/home/jovyan/work/intelimon-point-cloud-metrics-for-uc-climate-action-seed-project-scaling-science-driven-vegetat'`**

In [ ]:
DATASET_SOURCE = 'ndp'                  # 'ndp' | 'local'
DATASET_IDS: list[str] = []             # used when DATASET_SOURCE == 'ndp'
DATASETS_PATH = './datasets'            # used when DATASET_SOURCE == 'local'

if DATASET_SOURCE == 'ndp':
    if not DATASET_IDS:
        DATASET_IDS = list_datasets(limit=3)
        print(f'No DATASET_IDS set — using first 3 from catalog: {DATASET_IDS}')
    datasets = []
    for ds_id in DATASET_IDS:
        try:
            datasets.append(fetch_dataset(ds_id))
            print(f'  fetched {ds_id}')
        except Exception as e:
            print(f'  FAILED {ds_id}: {e}')

elif DATASET_SOURCE == 'local':
    datasets = load_local_datasets(DATASETS_PATH)
    datasets = _filter_dataset_entries(datasets)
    for d in datasets:
        print(f'  loaded {d.get("name") or d.get("title")}')

else:
    raise ValueError(f"DATASET_SOURCE must be 'ndp' or 'local', got {DATASET_SOURCE!r}")

print(f'\nLoaded {len(datasets)} dataset(s).')

## 4. Build a text document per dataset

Flatten each CKAN package into a single string — title, notes, tags, organization, resources, and extras.

In [ ]:
import io

# Cap how many data rows from any single resource we serialize into the bot's
# context. Set to None to include everything.
MAX_ROWS_PER_RESOURCE = None


def _resource_data_to_csv(rows: list[dict], max_rows: int | None) -> str:
    if not rows:
        return ''
    df = pd.DataFrame(rows)
    truncated = max_rows is not None and len(df) > max_rows
    if truncated:
        df = df.head(max_rows)
    buf = io.StringIO()
    df.to_csv(buf, index=False)
    text = buf.getvalue().strip()
    if truncated:
        text += f'\n... ({len(rows) - max_rows} more rows omitted)'
    return text


def dataset_to_document(pkg: dict) -> str:
    parts = []
    parts.append(f'Title: {pkg.get("title") or pkg.get("name", "")}')
    if pkg.get('notes'):
        parts.append(f'Description: {pkg["notes"]}')
    tags = [t.get('display_name') or t.get('name') for t in pkg.get('tags', [])]
    tags = [t for t in tags if t]
    if tags:
        parts.append('Tags: ' + ', '.join(tags))
    org = (pkg.get('organization') or {}).get('title')
    if org:
        parts.append(f'Organization: {org}')
    for extra in pkg.get('extras', []):
        k, v = extra.get('key'), extra.get('value')
        if k and v:
            parts.append(f'{k}: {v}')

    resource_summaries = []
    resource_blocks = []
    for r in pkg.get('resources', []):
        name = r.get('name') or r.get('description') or ''
        fmt = r.get('format') or ''
        resource_summaries.append(f'{name} ({fmt})'.strip())

        data = r.get('data')
        if isinstance(data, list) and data and isinstance(data[0], dict):
            csv_text = _resource_data_to_csv(data, MAX_ROWS_PER_RESOURCE)
            if csv_text:
                resource_blocks.append(f'Resource data — {name} ({fmt}):\n{csv_text}')

    if resource_summaries:
        parts.append('Resources: ' + '; '.join(resource_summaries))
    parts.extend(resource_blocks)

    return '\n'.join(p for p in parts if p)


documents = [dataset_to_document(d) for d in datasets]

print('--- First document preview ---')
print(documents[0][:2000] + ('...' if len(documents[0]) > 2000 else ''))
print(f'\n(full document length: {len(documents[0])} chars)')

## 5. Vectorize and store

**Embedding** — selected by `EMBED_MODEL_KEY` above. Default is NRP's `qwen3-embedding` (4096-dim).

**Storage** — ChromaDB, persisted under `./chroma_db/`. The `make_store(...)` factory leaves room to add other backends later without touching the rest of the notebook.

In [ ]:
_st_model = None  # cached sentence-transformers instance

def embed_texts(texts: list[str]) -> list[list[float]]:
    if embed_cfg is None:
        from sentence_transformers import SentenceTransformer
        global _st_model
        if _st_model is None:
            _st_model = SentenceTransformer('all-MiniLM-L6-v2')
        return _st_model.encode(texts, show_progress_bar=False).tolist()

    return call_embedding(
        texts=texts,
        model_name=embed_cfg['model'],
        api_key=embed_cfg['api_key'],
        base_url=embed_cfg['base_url'],
    )

# Figure out the vector dimension from one probe call.
EMBED_DIM = len(embed_texts(['probe'])[0])
print(f'Embedding dimension: {EMBED_DIM}')

In [ ]:
VECTOR_BACKEND = 'chroma'
COLLECTION = 'ndp_datasets'

store = make_store(VECTOR_BACKEND, collection=COLLECTION, dim=EMBED_DIM)
print(f'Using {VECTOR_BACKEND} backend: {store.mode}')

store.reset()

ids = [d.get('id') or d.get('name') for d in datasets]
metadatas = [
    {
        'name': d.get('name') or '',
        'title': d.get('title') or '',
        'organization': (d.get('organization') or {}).get('title') or '',
    }
    for d in datasets
]
embeddings = embed_texts(documents)

store.add(ids=ids, documents=documents, metadatas=metadatas, embeddings=embeddings)
print(f'Indexed {store.count()} dataset(s) into "{COLLECTION}".')

## 6. Chatbot

`DatasetBot` subclasses `BaseAgent`. Each turn it puts the full metadata for **every** dataset in the workspace into the context and calls the chat LLM — no top-k filtering, since the workspace only has a handful of datasets.

If you scale past that, replace `_format_context(self.documents, self.titles)` inside `chat_turn` with `_format_context_from_hits(retrieve(msg, k=...))` (see §6 for the retrieval helper).

In [ ]:
SYSTEM_PROMPT = (
    'You are an assistant that helps users explore the datasets in their workspace. '
    'You will be given the full metadata for each dataset. Answer the user\'s question '
    'using only that metadata — cite dataset titles when you refer to them. '
    'If the metadata does not cover the question, say so plainly.'
)

def _format_context(docs: list[str], titles: list[str]) -> str:
    blocks = []
    for i, (title, doc) in enumerate(zip(titles, docs), 1):
        blocks.append(f'[Dataset {i}: {title}]\n{doc}')
    return '\n\n'.join(blocks)


class DatasetBot(BaseAgent):
    name = 'DatasetBot'
    system_prompt = SYSTEM_PROMPT

    def __init__(self, model_name, api_key=None, base_url=None, temperature=0.2,
                 documents: list[str] = None, titles: list[str] = None):
        super().__init__(model_name=model_name, api_key=api_key, base_url=base_url, temperature=temperature)
        self.documents = documents or []
        self.titles = titles or []

    def chat_turn(self, msg: str) -> str:
        context = _format_context(self.documents, self.titles)

        if not self.history:
            self.add_system_message()

        user_content = f'Workspace datasets:\n\n{context}\n\nUser question: {msg}'
        self.add_user_message(user_content)

        reply = self.call_llm(history=self.history)
        self.add_assistant_message(reply)
        return reply


bot = DatasetBot(
    model_name=chat_cfg['model'],
    api_key=chat_cfg['api_key'],
    base_url=chat_cfg['base_url'],
    documents=documents,
    titles=[d.get('title') or d.get('name') or '' for d in datasets],
)
print(f'DatasetBot ready with {CHAT_MODEL_KEY} ({chat_cfg["model"]}) over {len(documents)} dataset(s).')

## 7. Ask it questions

Single-shot example:

In [ ]:
# List the metrics the bot can be questioned about.
# Pulls descriptions from dataset_keys_to_keep.csv and intersects them with
# the columns actually present in each loaded dataset's resources.
import pandas as pd

keys_df = pd.read_csv('dataset_keys_to_keep.csv')
descriptions = dict(zip(keys_df['Metrics'], keys_df['Description (simplified)'].fillna('')))

present = set()
for d in datasets:
    for r in d.get('resources', []):
        data = r.get('data')
        if isinstance(data, list) and data and isinstance(data[0], dict):
            present.update(data[0].keys())
        else:
            present.update(k for k in r.keys() if k not in ('name', 'format', 'data'))

available = [m for m in keys_df['Metrics'].dropna() if m in present]
missing = [m for m in present if m not in descriptions]

print(f'{len(available)} metric(s) available to question:\n')
for m in available:
    desc = descriptions.get(m, '').strip()
    print(f'  - {m}: {desc}' if desc else f'  - {m}')

if missing:
    print(f'\n(also present but not in dataset_keys_to_keep.csv: {sorted(missing)})')

In [ ]:
print(bot.chat_turn('What is the Intelimon Metric dataset about?'))

Interactive loop: type `exit` / `quit` to stop, `reset` to clear history.

In [ ]:
while True:
    try:
        msg = input('You: ').strip()
    except (EOFError, KeyboardInterrupt):
        break
    if not msg:
        continue
    if msg.lower() in {'exit', 'quit'}:
        break
    if msg.lower() == 'reset':
        bot.reset_history()
        print('(history cleared)')
        continue
    print('Bot:', bot.chat_turn(msg))